# 06 — EGFR キナーゼ解析チュートリアル
# EGFR Kinase Docking Analysis Tutorial

**ターゲット**: EGFR（ErbB1）キナーゼドメイン  
**事例**: Erlotinib (Type I / DFG-in) vs Lapatinib (Type I.5 / DFG-out)  
**PDB構造**: 1XKK (erlotinib, 2.6 Å) / 2ITX (lapatinib, 2.0 Å)

---

## このノートブックで学べること

1. PDB からの受容体・リガンド構造取得と前処理
2. グリッドボックスの自動定義
3. `VinaRunner` / `UniDockRunner` によるドッキング実行フロー
4. `ProLIFCalculator` を使った相互作用フィンガープリント解析
5. DFG-in vs DFG-out による結合ポケットの違いの可視化
6. 2D インタラクションマップの比較

> **Note**: ドッキング実行セクション（Section 4）は Vina または UniDock バイナリが必要です。  
> バイナリなしでも Section 5 以降の解析デモは実行可能です（結晶構造の配座を使用）。

In [ ]:
# CONFIG -----------------------------------------------------------------------
DATA_DIR = "../data/egfr"          # PDB ダウンロード先 / PDB download directory
RESULTS_DIR = "../results/egfr"    # ドッキング結果出力先
VINA_BINARY = "vina"               # 'vina' or 'unidock' or path to binary
# ------------------------------------------------------------------------------

## 1. セットアップ / Setup

In [ ]:
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, AllChem
from IPython.display import display

# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis import DockingResult, get_reader
from docking_analysis.preparation.receptor import (
    load_receptor, remove_solvent, select_protein, prepare_receptor
)
from docking_analysis.preparation.gridbox import gridbox_from_ligand, GridBox
from mdatools.docking.fingerprints.prolif import ProLIFCalculator
from mdatools.docking.visualization.interaction_map import draw_interaction_map, draw_interaction_grid
from mdatools.docking.analysis.properties import calculate_properties
from mdatools.docking.clustering.chemical import compute_fp_matrix, cluster_by_butina

RDLogger.DisableLog("rdApp.warning")

data_dir = Path(DATA_DIR)
results_dir = Path(RESULTS_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
print("Setup complete.")

## 2. PDB 構造の取得 / Download PDB Structures

| PDB ID | リガンド | 結合様式 | 分解能 |
|--------|---------|---------|--------|
| **1XKK** | Erlotinib (Tarceva) | Type I / DFG-in | 2.6 Å |
| **2ITX** | Lapatinib (Tykerb) | Type I.5 / DFG-out | 2.0 Å |

In [ ]:
PDB_IDS = ["1XKK", "2ITX"]

for pdb_id in PDB_IDS:
    dest = data_dir / f"{pdb_id.lower()}.pdb"
    if not dest.exists() or dest.stat().st_size == 0:
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        print(f"Downloading {pdb_id}...", end=" ")
        urllib.request.urlretrieve(url, dest)
        print(f"→ {dest}")
    else:
        print(f"{pdb_id}: already exists ({dest})")

In [ ]:
def list_hetatm_residues(pdb_path: Path, min_atoms: int = 6) -> list[dict]:
    """Parse HETATM records and return unique non-water residues."""
    WATER_CODES = {"HOH", "WAT", "H2O", "DOD", "D2O"}
    seen = {}
    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("HETATM"):
                continue
            res_name = line[17:20].strip()
            chain = line[21].strip()
            seq_id = line[22:26].strip()
            key = (chain, res_name, seq_id)
            if res_name not in WATER_CODES:
                seen[key] = seen.get(key, 0) + 1
    result = [
        {"chain": k[0], "residue": k[1], "seqid": k[2], "n_atoms": v}
        for k, v in seen.items()
        if v >= min_atoms
    ]
    return sorted(result, key=lambda x: -x["n_atoms"])

for pdb_id in PDB_IDS:
    residues = list_hetatm_residues(data_dir / f"{pdb_id.lower()}.pdb")
    print(f"\n{pdb_id} — 主要リガンド候補 (≥6 heavy atoms):")
    for r in residues:
        print(f"  Chain {r['chain']}: {r['residue']:>4}  seqid={r['seqid']:>5}  atoms={r['n_atoms']}")

## 3. リガンド・受容体の抽出と準備
## Extract Ligands and Prepare Receptor

上の出力を確認して `LIGAND_CODES` を設定してください。  
Check the output above and set `LIGAND_CODES` accordingly.

- 1XKK: erlotinib の残基コードを入力（例: `"ERL"`）
- 2ITX: lapatinib の残基コードを入力（例: `"LPT"` / `"LAP"`）

In [ ]:
# ↓ 上のセルの出力を見て残基コードを設定 / Set residue codes from output above
LIGAND_CODES = {
    "1XKK": "FMM",   # erlotinib (RCSB code)
    "2ITX": "ANP",   # lapatinib (RCSB code)
}

In [ ]:
def extract_ligand_mol(pdb_path: Path, res_code: str, chain: str = "A") -> Chem.Mol | None:
    """Extract a ligand from PDB HETATM records and return an RDKit Mol."""
    with open(pdb_path) as f:
        lines = f.readlines()

    hetatm = [
        l for l in lines
        if l.startswith("HETATM")
        and l[17:20].strip() == res_code
        and (chain == "*" or l[21].strip() == chain)
    ]
    if not hetatm:
        print(f"  WARNING: residue {res_code} not found in {pdb_path.name}")
        return None

    pdb_block = "".join(hetatm) + "END\n"
    mol = Chem.MolFromPDBBlock(pdb_block, removeHs=True, sanitize=True)
    if mol is None:
        print(f"  WARNING: RDKit could not parse {res_code}")
    return mol


ligand_mols = {}
for pdb_id, res_code in LIGAND_CODES.items():
    mol = extract_ligand_mol(data_dir / f"{pdb_id.lower()}.pdb", res_code)
    if mol is not None:
        # Name the molecule
        mol.SetProp("mol_name", res_code)
        mol.SetProp("pdb_id", pdb_id)
        mol.SetProp("pose_rank", "1")
        mol.SetProp("docking_score", "0.0")  # placeholder (crystal pose)
        ligand_mols[pdb_id] = mol
        print(f"{pdb_id} [{res_code}]: {mol.GetNumAtoms()} heavy atoms")
    else:
        print(f"{pdb_id} [{res_code}]: failed — check the residue code above")

# Display structures
if len(ligand_mols) == 2:
    mols_to_draw = list(ligand_mols.values())
    legends = [f"{pid}\n{LIGAND_CODES[pid]}" for pid in ligand_mols]
    img = Draw.MolsToGridImage(mols_to_draw, molsPerRow=2, subImgSize=(400, 300), legends=legends)
    display(img)

In [ ]:
# 受容体準備: 溶媒・リガンドを除去してタンパク質のみを抽出
# Receptor preparation: remove solvent/ligand, keep only protein

receptor_paths = {}
for pdb_id in PDB_IDS:
    raw_pdb = data_dir / f"{pdb_id.lower()}.pdb"
    out_pdb = data_dir / f"{pdb_id.lower()}_receptor.pdb"

    if not out_pdb.exists():
        print(f"Preparing {pdb_id} receptor...", end=" ")
        prepare_receptor(raw_pdb, out_pdb)
        print(f"→ {out_pdb}")
    else:
        print(f"{pdb_id} receptor ready: {out_pdb}")

    receptor_paths[pdb_id] = out_pdb

## 4. グリッドボックスの設定
## Grid Box Setup

結晶リガンドの重心から自動的にドッキングのグリッドボックスを定義します。  
Automatically define docking grid boxes from the crystal ligand centroid.

In [ ]:
gridboxes = {}
for pdb_id, mol in ligand_mols.items():
    # 3D 座標が必要な場合は crystal structure の座標をそのまま使用
    if mol.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())

    grid = gridbox_from_ligand(mol, padding=5.0)
    gridboxes[pdb_id] = grid

    print(f"\n{pdb_id} グリッドボックス:")
    print(f"  中心 (Å): ({grid.center[0]:.2f}, {grid.center[1]:.2f}, {grid.center[2]:.2f})")
    print(f"  サイズ (Å): ({grid.size[0]:.2f}, {grid.size[1]:.2f}, {grid.size[2]:.2f})")

# DFG-in (1XKK) vs DFG-out (2ITX) の結合部位中心の差
if len(gridboxes) == 2:
    c1 = gridboxes["1XKK"].center
    c2 = gridboxes["2ITX"].center
    shift = sum((a - b)**2 for a, b in zip(c1, c2)) ** 0.5
    print(f"\n結合部位中心のシフト (1XKK vs 2ITX): {shift:.2f} Å")
    print("※ DFG-in → DFG-out の構造変化が大きいほどシフトが大きくなります")

## 5. ドッキング実行
## Docking Execution

> ⚠️ このセクションは Vina または UniDock バイナリが必要です。  
> バイナリがない場合は Section 6（解析デモ）に進んでください。

**CPU 環境 (Vina):**
```bash
pip install vina  # or conda install -c conda-forge autodock-vina
```

**GPU 環境 (UniDock):**
```bash
docker compose -f docker/docker-compose.yml --profile gpu up
```

In [ ]:
# リガンド PDBQT 変換 (meeko 使用)
# Ligand PDBQT conversion using meeko
from docking_analysis.preparation.ligand import prepare_for_vina

pdbqt_paths = {}
for pdb_id, mol in ligand_mols.items():
    # 3D 座標の確認・生成
    # addCoords=True: 既存コンフォーマーがあれば Hs 座標も生成 (crystal pose 対応)
    mol_h = Chem.AddHs(mol, addCoords=True)
    if mol_h.GetNumConformers() == 0:
        AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3())
        AllChem.MMFFOptimizeMolecule(mol_h)

    out_pdbqt = data_dir / f"{pdb_id.lower()}_{LIGAND_CODES[pdb_id].lower()}.pdbqt"
    try:
        prepare_for_vina(mol_h, out_pdbqt)
        pdbqt_paths[pdb_id] = out_pdbqt
        print(f"{pdb_id}: → {out_pdbqt}")
    except Exception as e:
        print(f"{pdb_id}: PDBQT 変換エラー — {e}")


In [ ]:
# ドッキング実行 (Vina バイナリが必要 / Requires Vina binary)
# ---------------------------------------------------------------
# 以下は実行例です。バイナリがない場合は次のセルをスキップして
# Section 6 のデモデータを使ってください。
# ---------------------------------------------------------------

import shutil
if shutil.which(VINA_BINARY) is None:
    print(f"⚠ '{VINA_BINARY}' not found in PATH — skipping docking.")
    print("Section 6 uses crystal structure poses as demo data instead.")
else:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
    from docking_analysis.docking import get_runner
    from docking_analysis.docking.runner import VinaRunConfig

    runner = get_runner(VINA_BINARY if VINA_BINARY != "vina" else "vina",
                        exhaustiveness=8, num_modes=9)

    # 1XKK の受容体に対してクリスタルリガンドを再ドッキング
    receptor_pdbqt = data_dir / "1xkk_receptor.pdbqt"  # meeko で別途変換
    if "1XKK" in pdbqt_paths and receptor_pdbqt.exists():
        result = runner.run(
            ligand_pdbqt=pdbqt_paths["1XKK"],
            receptor_pdbqt=receptor_pdbqt,
            grid=gridboxes["1XKK"],
            output_pdbqt=results_dir / "1xkk_erlotinib_out.pdbqt",
        )
        print(f"Top score: {result.scores[0]:.2f} kcal/mol")
        print(f"Poses    : {len(result.poses)}")

## 6. 結合様式の比較解析 (結晶構造ポーズ使用)
## Binding Mode Analysis (Using Crystal Poses)

以下では**結晶構造から抽出したポーズ**を `DockingResult` として利用し、
ライブラリの解析機能をデモします。実際のドッキング結果がある場合は
`get_reader(backend).read(path)` で読み込んだ `result` を使ってください。

In [ ]:
# 結晶ポーズから DockingResult を構築
# Build DockingResult from crystal structure poses

crystal_results = {}
for pdb_id, mol in ligand_mols.items():
    crystal_results[pdb_id] = DockingResult(
        poses=[mol],
        scores=[0.0],               # crystal pose: no docking score
        source_file=data_dir / f"{pdb_id.lower()}.pdb",
        backend="crystal",
        metadata={"source": "crystal_structure"},
    )

print("DockingResult from crystal poses:")
for pid, res in crystal_results.items():
    print(f"  {pid}: {len(res.poses)} pose(s), ligand {LIGAND_CODES[pid]}")

### 6.1 ProLIF 相互作用フィンガープリント
### ProLIF Interaction Fingerprints

EGFR 結合部位の**ヒンジ残基・DFG ループ・ゲートキーパー残基**との相互作用を解析します。

In [ ]:
calculator = ProLIFCalculator()

fp_results = {}
for pdb_id, result in crystal_results.items():
    receptor_pdb = receptor_paths[pdb_id]
    protein_mol = Chem.MolFromPDBFile(str(receptor_pdb), removeHs=False)
    if protein_mol is None:
        print(f"  {pdb_id}: receptor load failed — skip")
        continue

    print(f"\n{pdb_id} — ProLIF fingerprint...")
    fp_df = calculator.calculate(result.poses, protein_mol, show_progress=True)
    fp_results[pdb_id] = fp_df

    # 接触のあった残基と相互作用タイプを表示
    active_cols = fp_df.columns[fp_df.any()].tolist()
    print(f"  検出された相互作用 ({len(active_cols)}件):")
    for col in active_cols:
        print(f"    {col}")

### 6.2 インタラクションマップの比較
### Interaction Map Comparison

Erlotinib (1XKK / Type I) と Lapatinib (2ITX / Type I.5) の
2D インタラクションマップを並べて比較します。

**注目ポイント:**
- Erlotinib: ヒンジ残基（Met769 相当）への H 結合 2 点
- Lapatinib: DFG-out ポケットへの追加疎水接触 + Thr/Lys との新規接触

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

pdb_list = ["1XKK", "2ITX"]
ligand_names = {"1XKK": "Erlotinib (Type I, DFG-in)",
                "2ITX": "Lapatinib (Type I.5, DFG-out)"}

for ax, pdb_id in zip(axes, pdb_list):
    if pdb_id not in fp_results:
        ax.set_title(f"{pdb_id}: data not available")
        continue

    fp_df = fp_results[pdb_id]
    mol = ligand_mols[pdb_id]
    active_cols = fp_df.columns[fp_df.any()].tolist()

    if active_cols:
        draw_interaction_map(
            mol=mol,
            interactions=active_cols,
            ax=ax,
            title=f"{pdb_id}\n{ligand_names[pdb_id]}"
        )
    else:
        ax.set_title(f"{pdb_id}: no interactions detected")

plt.tight_layout()
out_fig = results_dir / "egfr_interaction_map_comparison.png"
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_fig}")

### 6.3 分子物性と Ligand Efficiency の比較
### Molecular Properties and Ligand Efficiency Comparison

In [ ]:
from docking_analysis.analysis.properties import add_properties_to_df

rows = []
for pdb_id, mol in ligand_mols.items():
    props = calculate_properties(mol)
    rows.append({
        "compound": f"{LIGAND_CODES[pdb_id]} ({pdb_id})",
        "binding_type": "Type I (DFG-in)" if pdb_id == "1XKK" else "Type I.5 (DFG-out)",
        **props,
    })

props_df = pd.DataFrame(rows).set_index("compound")
display(props_df.round(3))

### 6.4 結合様式サマリー
### Binding Mode Summary

| 特徴 | Erlotinib (1XKK) | Lapatinib (2ITX) |
|------|-----------------|------------------|
| DFG ループ状態 | **DFG-in** (active) | **DFG-out** (inactive) |
| 結合モード分類 | Type I | Type I.5 |
| ヒンジ H 結合 | Met769 に 2 点 | Met769 に 1 点 |
| 追加疎水ポケット | なし | DFG-out pocket を占有 |
| 選択性プロファイル | 広い（多くのキナーゼ阻害） | HER2 併用阻害 |
| 解離速度 $k_{off}$ | 速い | 遅い（long residence time）|

## 7. 仮想スクリーニングへの応用
## Application to Virtual Screening

EGFR 阻害剤のデカイン化合物ライブラリを用いた仮想スクリーニングを想定した場合のワークフロー例です。
実際のライブラリ CSV を用意して `smiles` 列を読み込んでください。

> ここでは erlotinib / lapatinib とその類縁体（ダミーデータ）で手順を示します。

In [ ]:
# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.preparation.ligand import prepare_batch_for_vina
from docking_analysis.preparation.normalize import standardize_df
from mdatools.docking.selection.filters import ScoreFilter, apply_filters
from mdatools.docking.analysis.properties import add_properties_to_df
from mdatools.docking.clustering.chemical import cluster_by_butina

# ダミーライブラリ (erlotinib / lapatinib + fictitious analogs)
# 実際には ChEMBL や Enamine REAL から取得
demo_library = pd.DataFrame({
    "compound_id": ["ERL-001", "ERL-002", "LAP-001", "LAP-002", "DUMMY-001"],
    "smiles": [
        "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1",   # erlotinib
        "C#Cc1cccc(Nc2ncnc3cc(OC)c(OC)cc23)c1",          # erlotinib analog
        "CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1",  # lapatinib
        "CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OC)c(Cl)c4)c3c2)o1",            # lapatinib analog
        "c1ccc(-c2ccccc2)cc1",  # biphenyl (negative control)
    ],
})

# 分子標準化
std_df = standardize_df(demo_library, smiles_col="smiles")
print(f"標準化後: {len(std_df)} 化合物")

# 物性追加
std_df = add_properties_to_df(std_df, smiles_col="smiles")
display(std_df[["compound_id", "smiles", "mw", "logp", "hbd", "hba", "qed"]].round(2))

In [ ]:
# 化学的多様性クラスタリング (Butina)
mols_for_cluster = [
    Chem.MolFromSmiles(smi) for smi in std_df["smiles"] if Chem.MolFromSmiles(smi) is not None
]
labels = cluster_by_butina(mols_for_cluster, threshold=0.4)
std_df["cluster"] = labels

print("クラスター割り当て:")
display(std_df[["compound_id", "cluster"]])
print(f"\nユニークなクラスター数: {std_df['cluster'].nunique()}")
print("→ Erlotinib系 / Lapatinib系 / ネガティブコントロールの3クラスターが期待される")

## 8. 次のステップ / Next Steps

このノートブックで示したワークフロー:

```
PDB ダウンロード → 受容体準備 → グリッドボックス定義
  → [ドッキング実行] → ProLIF 解析 → 結合様式比較
  → 物性計算 → クラスタリング → 化合物選択
```

**発展的な解析:**
- `compute_consensus_score()` — 1XKK と 2ITX に対する consensus scoring
- `validate_poses_posebusters()` — ポーズ品質ゲート
- `select_maxmin_diverse()` — 最終候補の多様性選択
- `notebooks/07_gpcr_a2a.ipynb` — GPCR 解析チュートリアル（次のノートブック）

---

**関連 Issue:**
- Issue #75: このノートブック
- Issue #76: GPCR A2A チュートリアル
- Issue #77: EGFR 共有結合ドッキング